# OpenCanvas Agent Testing Ground

This notebook provides a testing environment for OpenCanvas agents. You can use it to:
- Initialize the agent graph
- Create test states
- Execute the workflow with different inputs
- Analyze the results

In [1]:
# Import required modules
import os
import sys
import json
import asyncio
import dotenv
from pprint import pprint
from typing import Dict, Any, List

# Add the current directory to the path so we can import modules
sys.path.append(os.path.abspath('..'))

# Load environment variables
dotenv.load_dotenv()

# Import agent modules
from agents.src.open_canvas.index import graph as open_canvas_graph
from agents.src.open_canvas.state import OpenCanvasGraphState
from langchain_core.messages import HumanMessage, AIMessage
from shared.src.types import ArtifactV3, ArtifactContent


%load_ext autoreload
%autoreload 2


/Users/kaitlynhennacy/Library/Caches/pypoetry/virtualenvs/storia-backend-18Z1xCOI-py3.12/lib/python3.12/site-packages/pydantic/_internal/_config.py:345: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


## Graph Initialization

Initialize the OpenCanvas agent graph and verify its structure.

In [8]:
# Print graph information
print(f"Graph name: {open_canvas_graph.name}")
print(f"Graph nodes: {open_canvas_graph.nodes}")

# Helper function to execute the graph
async def run_graph(state: Dict[str, Any]):
    """Run the OpenCanvas graph with the given state."""
    config = {}
    result = await open_canvas_graph.ainvoke(state, config)
    return result

Graph name: open_canvas
Graph nodes: {'__start__': <langgraph.pregel.read.PregelNode object at 0x141d861b0>, 'generate_path': <langgraph.pregel.read.PregelNode object at 0x141d86090>, 'reply_to_general_input': <langgraph.pregel.read.PregelNode object at 0x141d865d0>, 'rewriteArtifact': <langgraph.pregel.read.PregelNode object at 0x141d86e10>, 'rewrite_artifact_theme': <langgraph.pregel.read.PregelNode object at 0x141d87650>, 'rewriteCodeArtifactTheme': <langgraph.pregel.read.PregelNode object at 0x141d87ad0>, 'updateArtifact': <langgraph.pregel.read.PregelNode object at 0x141d87410>, 'updateHighlightedText': <langgraph.pregel.read.PregelNode object at 0x141d870b0>, 'generateArtifact': <langgraph.pregel.read.PregelNode object at 0x141d873b0>, 'customAction': <langgraph.pregel.read.PregelNode object at 0x141d87bf0>, 'generateFollowup': <langgraph.pregel.read.PregelNode object at 0x141d87710>, 'cleanState': <langgraph.pregel.read.PregelNode object at 0x141d87920>, 'generateTitle': <langgr

## Test States

Create various test states to send to the agent.

In [3]:
# Create a simple text generation state
def create_text_generation_state():
    return {
        "_messages": [
            HumanMessage(content="Write a short blog post about artificial intelligence.")
        ],
        "messages": [
            HumanMessage(content="Write a short blog post about artificial intelligence.")
        ]
    }

# Create a code generation state
def create_code_generation_state():
    return {
        "_messages": [
            HumanMessage(content="Write a Python function to calculate Fibonacci numbers recursively.")
        ],
        "messages": [
            HumanMessage(content="Write a Python function to calculate Fibonacci numbers recursively.")
        ]
    }

# Create a state with an existing artifact for rewriting
def create_rewrite_artifact_state():
    # Create a sample artifact
    artifact_content = ArtifactContent(
        type="markdown",
        full_markdown="# Sample Blog Post\n\nThis is a sample blog post about technology.\n\n## Introduction\n\nTechnology has changed our lives in many ways.",
        language="markdown"
    )
    
    artifact = ArtifactV3(
        current_index=1,
        contents=[artifact_content]
    )
    
    return {
        "_messages": [
            HumanMessage(content="Rewrite this to be more engaging and add a section about AI.")
        ],
        "messages": [
            HumanMessage(content="Rewrite this to be more engaging and add a section about AI.")
        ],
        "artifact": artifact
    }

# Create a state for web search
def create_web_search_state():
    return {
        "_messages": [
            HumanMessage(content="What are the latest developments in quantum computing?")
        ],
        "messages": [
            HumanMessage(content="What are the latest developments in quantum computing?")
        ],
        "web_search_enabled": False
    }

## Test Execution

Execute the agent workflow with different test states.

In [17]:
# Test the text generation state
print("\n==== Testing Text Generation ====")
text_state = create_text_generation_state()
import concurrent.futures
with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
    futures = [executor.submit(run_graph, text_state)]
    results = []
    for future in concurrent.futures.as_completed(futures):
        results.append(future.result())
    text_result = results[0]
print("\nResult of text generation:")
pprint(text_result)


==== Testing Text Generation ====

Result of text generation:
<coroutine object run_graph at 0x146db0ba0>


/var/folders/r2/skd5m_bn33xb75hyffgqnmz80000gn/T/ipykernel_26314/583099688.py:10: RuntimeWarning: coroutine 'run_graph' was never awaited
  text_result = results[0]


In [ ]:
# Test the code generation state
print("\n==== Testing Code Generation ====")
code_state = create_code_generation_state()
code_result = await run_graph(code_state)
print("\nResult of code generation:")
pprint(code_result)

In [ ]:
# Test the rewrite artifact state
print("\n==== Testing Artifact Rewriting ====")
rewrite_state = create_rewrite_artifact_state()
rewrite_result = await run_graph(rewrite_state)
print("\nResult of artifact rewriting:")
pprint(rewrite_result)

In [ ]:
# Test the web search state
print("\n==== Testing Web Search ====")
try:
    search_state = create_web_search_state()
    search_result = await run_graph(search_state)
    print("\nResult of web search:")
    pprint(search_result)
except Exception as e:
    print(f"Error during web search: {e}")

## Custom Test State

Create a custom state to test specific functionality.

In [ ]:
# Create a custom test state
def create_custom_test_state():
    # Define your custom state here
    return {
        "_messages": [
            HumanMessage(content="Your custom message here")
        ],
        "messages": [
            HumanMessage(content="Your custom message here")
        ],
        # Add any additional state properties needed
    }

# Test with custom state
print("\n==== Testing Custom State ====")
custom_state = create_custom_test_state()
custom_result = await run_graph(custom_state)
print("\nResult of custom test:")
pprint(custom_result)

## Result Analysis

Helper functions to analyze and display the results.

In [ ]:
def display_artifact_content(result):
    """Display the artifact content in a readable format."""
    if not result or 'artifact' not in result:
        print("No artifact in result")
        return
    
    artifact = result['artifact']
    if not artifact or 'contents' not in artifact or not artifact['contents']:
        print("No contents in artifact")
        return
    
    content = artifact['contents'][-1]  # Get the latest content
    
    print(f"Artifact Type: {content.get('type', 'unknown')}")
    print(f"Language: {content.get('language', 'unknown')}")
    print("\nContent:")
    
    if content.get('type') == 'markdown':
        print(content.get('full_markdown', 'No markdown content'))
    elif content.get('type') == 'code':
        print(content.get('code', 'No code content'))
    else:
        print(json.dumps(content, indent=2))

# Display the artifact content for each result
print("\n==== Text Generation Artifact ====")
display_artifact_content(text_result)

print("\n==== Code Generation Artifact ====")
display_artifact_content(code_result)

print("\n==== Rewritten Artifact ====")
display_artifact_content(rewrite_result)